# Assignment 1 — Finding Assistant (RAG Agent)

Walkthrough of the RAG pipeline: load → chunk → embed → retrieve → answer → structured finding, plus thread-based short-term memory.

**Prerequisite:** a populated `.env` (copy from `.env.example`) with valid Azure OpenAI credentials.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `import src.*` works from notebooks/.
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Build (or load) the vector store

Loads `data/BonBon FAQ.pdf`, chunks it, embeds the chunks, and persists them to `.chroma/`. Idempotent — re-running loads the existing index.

In [ ]:
from src.vector_store import build_vectorstore
from src.loaders import load_pdf
from src.splitters import split_documents

docs = load_pdf()
chunks = split_documents(docs)
print(f"Loaded {len(docs)} pages -> {len(chunks)} chunks")
print("\n--- First chunk preview ---")
print(chunks[0].page_content[:400])

store = build_vectorstore()
print("\nVector store ready.")

## 2. Retrieve relevant context

In [ ]:
from src.retriever import get_retriever, format_docs

hits = get_retriever(k=3).invoke("How do I reset my password?")
print(format_docs(hits))

## 3. Ask the agent (grounded answer + tool use)

In [ ]:
from src.agent import answer_question

for q in [
    "How do I reset my password?",
    "My internet connection is not working. How do I troubleshoot it?",
]:
    print(f"Q: {q}")
    print(f"A: {answer_question(q, thread_id='nb-demo')}\n")

## 4. Extract a structured Finding (Pydantic / schema-based)

In [ ]:
from src.agent import extract_finding

q = "How do I reset my password?"
a = answer_question(q, thread_id='nb-demo')
finding = extract_finding(q, a)
print(finding.model_dump_json(indent=2))

## 5. Short-term memory: same thread vs different thread

In [ ]:
follow_up = "Can you summarize what we just discussed?"

print("SAME thread ('nb-demo') — should recall the password topic:")
print(answer_question(follow_up, thread_id='nb-demo'))

print("\nDIFFERENT thread ('fresh') — should NOT recall it:")
print(answer_question(follow_up, thread_id='fresh'))